## TEST AOSS INDEX

In [4]:
session

Session(region_name='us-east-1')

In [8]:
import boto3

# 1. Verificar qué role estás usando
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"Role ARN: {identity['Arn']}")

# 2. Verificar si tienes permisos IAM para AOSS
aoss_client = boto3.client('opensearchserverless')
try:
    collections = aoss_client.list_collections()
    print(f"Collections: {collections['collectionSummaries']}")
except Exception as e:
    print(f"Error listando collections: {e}")


Role ARN: arn:aws:sts::697682206292:assumed-role/sgmkr-notebook-tfm-hymm-rec-ml-iar-dev/SageMaker
Collections: [{'id': 'f3cprbwt2yb9t7xvm84a', 'name': 'hymmrec-items-vectors', 'status': 'ACTIVE', 'arn': 'arn:aws:aoss:us-east-1:697682206292:collection/f3cprbwt2yb9t7xvm84a', 'kmsKeyArn': 'auto', 'collectionGroupName': 'hymmrec-vectors-cg'}]


In [24]:
pip install opensearch-py requests-aws4auth

Note: you may need to restart the kernel to use updated packages.


In [27]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3

region = "us-east-1"
service = "aoss"

credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    service,
    session_token=credentials.token
)

host = "f3cprbwt2yb9t7xvm84a.us-east-1.aoss.amazonaws.com"

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

# Saltar client.info() — no existe en AOSS
# Ir directo a crear el índice
index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512
        }
    },
    "mappings": {
        "properties": {
            "item_idx": {"type": "integer"},
            "movie_id": {"type": "integer"},
            "title": {"type": "text"},
            "genres": {"type": "keyword"},
            "synopsis": {"type": "text"},
            "release_year": {"type": "integer"},
            "poster_path": {"type": "keyword"},
            "director": {"type": "keyword"},
            "item_embedding": {
                "type": "knn_vector",
                "dimension": 64,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "parameters": {
                        "ef_construction": 512,
                        "m": 16
                    }
                }
            },
            "attention_weights": {
                "type": "object",
                "properties": {
                    "category": {"type": "float"},
                    "text": {"type": "float"},
                    "image": {"type": "float"}
                }
            },
            "indexed_at": {"type": "date"}
        }
    }
}

try:
    response = client.indices.create(index="hymmrec-items-vectors", body=index_body)
    print("Índice creado:", response)
except Exception as e:
    print(f"Error: {e}")


Índice creado: {'acknowledged': True, 'shards_acknowledged': False, 'index': 'hymmrec-items-vectors'}


In [28]:
# Eliminar índice
response = client.indices.delete(index="hymmrec-items-vectors")
print("Índice eliminado:", response)


Índice eliminado: {'acknowledged': True}
